In [1]:
# Lirary load
install.packages(c("dplyr", "caret","glmnet"),repos='http://cran.us.r-project.org') 
library(dplyr)
library(caret) 
library(glmnet)

Warning message in install.packages(c("dplyr", "caret", "glmnet"), repos = "http://cran.us.r-project.org"):
“installation of package ‘glmnet’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: ggplot2

Loading required package: lattice

Loading required package: Matrix

Loaded glmnet 4.1-8



In [2]:
#1. Load dataset
train = read.csv("train.csv")
test = read.csv("test.csv")
cat("Raw training data", nrow(train), "rows", ncol(train),"cols\n")
cat("Raw test data", nrow(test), "rows", ncol(test),"cols\n")

Raw training data 630000 rows 13 cols
Raw test data 270000 rows 12 cols


In [7]:
#1. Model evaluation metric :rMSE

calculate_rMSE = function(actual,predicted){
  sqrt(mean((actual-predicted)^2))
}

#2. One-hot encode function 

onehot_encode = function(data, target_col = "exam_score") {
    # create copy
    df = data
    # identify all cols
    all_cols = setdiff(names(df), target_col)
    # identify categorical cols 
    categorical_cols = names(df)[sapply(df, function(x) is.character(x) | is.factor(x))]
    categorical_cols = setdiff(categorical_cols, target_col)
    # identify numerical cols
    numerical_cols = setdiff(all_cols, categorical_cols)
    # print summary
    cat(sprintf("Total features (excluding target): %d\n", length(all_cols)))
    cat(sprintf("  Categorical: %d (%s)\n", length(categorical_cols), 
                paste(categorical_cols, collapse = ", ")))
    cat(sprintf("  Numerical: %d (%s)\n", length(numerical_cols), 
                paste(numerical_cols, collapse = ", ")))

    # create dummy variables for categorical features
    dummy_model = NULL
    
    if(length(categorical_cols) > 0) {
        cat("\nCreating one-hot encoded features...\n")
        
        #  create formula for dummy variables
        dummy_formula = as.formula(paste("~", paste(categorical_cols, collapse = " + ")))
        
        # fit dummy model
        dummy_model = dummyVars(dummy_formula, data = df, fullRank = TRUE)
        
        # create dummy variables
        dummy_data = predict(dummy_model, newdata = df)
        dummy_df = as.data.frame(dummy_data)
        
        cat(sprintf("Number of dummy variables created: %d\n", ncol(dummy_df)))
        
        # combine dummy variables + numerical features + target (if exists)
        if(target_col %in% names(df)) {
            # Training/validation data (has target)
            df_final = cbind(dummy_df, 
                           df[, numerical_cols, drop = FALSE], 
                           df[, target_col, drop = FALSE])
        } else {
            # test data (no target)
            df_final = cbind(dummy_df, 
                           df[, numerical_cols, drop = FALSE])
        }
        
    } else {
        # no categorical features
        cat("\nNo categorical features found - using numerical features only\n")
        
        if(target_col %in% names(df)) {
            df_final = cbind(df[, numerical_cols, drop = FALSE], 
                           df[, target_col, drop = FALSE])
        } else {
            df_final = df[, numerical_cols, drop = FALSE]
        }
    }
    
    # print final summary
    cat(sprintf("\nFinal dataset dimensions: %d rows × %d columns\n", 
                nrow(df_final), ncol(df_final)))
    
    if(target_col %in% names(df_final)) {
        cat(sprintf("Features (excluding target): %d\n", ncol(df_final) - 1))
    } else {
        cat(sprintf("Features: %d\n", ncol(df_final)))
    }
    
    # return results
    result = list(
        data = df_final,
        dummy_model = dummy_model,
        categorical_cols = categorical_cols,
        numerical_cols = numerical_cols
    )
    
    return(result)
}

cat("\nFunctions loaded successfully\n")


Functions loaded successfully


In [8]:
# Process Train data 
train_clean = onehot_encode(
    data = train,
    target_col = "exam_score")

# Extract results
train_encoded = train_clean$data
dummy_model = train_clean$dummy_model
categorical_cols = train_clean$categorical_cols
numerical_cols = train_clean$numerical_cols
cat("\nEncoded training data:", nrow(train_encoded), "rows ×", ncol(train_encoded), "columns\n")

# Process Test data
if(!is.null(dummy_model)) {
    # Apply the same dummy model to test data
    test_dummy_data = predict(dummy_model, newdata = test)
    test_dummy_df = as.data.frame(test_dummy_data)
    
    # Combine with numerical features (no target column in test)
    test_encoded = cbind(test_dummy_df, test[, numerical_cols, drop = FALSE])
    
    cat("Test data encoded:", nrow(test_encoded), "rows ×", ncol(test_encoded), "columns\n")
    
} else {
    # No categorical columns - use numerical features only
    test_encoded = test[, numerical_cols, drop = FALSE]
    cat("Test data (numerical only):", nrow(test_encoded), "rows ×", ncol(test_encoded), "columns\n")
}

# Verify column alignment
cat("\n=== VERIFICATION ===\n")
train_features = ncol(train_encoded) - 1  # Minus target column
test_features = ncol(test_encoded)
if(test_features == train_features) {
    cat("✓ Column counts match perfectly!\n")
    cat(sprintf("  Train features: %d\n", train_features))
    cat(sprintf("  Test features: %d\n", test_features))
} else {
    warning("✗ Column mismatch detected!")
    cat(sprintf("  Train features: %d\n", train_features))
    cat(sprintf("  Test features: %d\n", test_features))
    cat(sprintf("  Difference: %d columns\n", abs(train_features - test_features)))
}

# Check column names alignment
cat("\n=== COLUMN NAME CHECK ===\n")
train_feature_names = setdiff(names(train_encoded), "exam_score")
test_feature_names = names(test_encoded)

if(all(test_feature_names %in% train_feature_names)) {
    cat("All test columns exist in training data\n")
} else {
    missing_in_train = setdiff(test_feature_names, train_feature_names)
    cat("Test has columns not in train:", paste(missing_in_train, collapse = ", "), "\n")
}
if(all(train_feature_names %in% test_feature_names)) {
    cat("All training features present in test data\n")
} else {
    missing_in_test = setdiff(train_feature_names, test_feature_names)
    cat("Train has columns not in test:", paste(missing_in_test, collapse = ", "), "\n")
}

Total features (excluding target): 12
  Categorical: 7 (gender, course, internet_access, sleep_quality, study_method, facility_rating, exam_difficulty)
  Numerical: 5 (id, age, study_hours, class_attendance, sleep_hours)

Creating one-hot encoded features...
Number of dummy variables created: 19

Final dataset dimensions: 630000 rows × 25 columns
Features (excluding target): 24

Encoded training data: 630000 rows × 25 columns
Test data encoded: 270000 rows × 24 columns

=== VERIFICATION ===
✓ Column counts match perfectly!
  Train features: 24
  Test features: 24

=== COLUMN NAME CHECK ===
All test columns exist in training data
All training features present in test data


In [9]:
# Split 70/30 train-validation
set.seed(123)
train_index = createDataPartition(train_encoded$exam_score, p = 0.7, list = FALSE) #

train_data = train_encoded[train_index, ]
validation_data = train_encoded[-train_index, ]
cat("\nData split complete:\n")
cat("Training set:", nrow(train_data), "rows ×", ncol(train_data), "columns\n")
cat("Validation set:", nrow(validation_data), "rows ×", ncol(validation_data), "columns\n")



Data split complete:
Training set: 441002 rows × 25 columns
Validation set: 188998 rows × 25 columns


### LASSO 

In [ ]:
# 1.Train LASSO model(lambda 1se) 
X_train = as.matrix(train_data[, -which(names(train_data) == "exam_score")])
y_train = train_data$exam_score

X_val = as.matrix(validation_data[, -which(names(validation_data) == "exam_score")])
y_val = validation_data$exam_score

# 2.Cross validation for LASSO 
set.seed(123)
cv_lasso = cv.glmnet(X_train, y_train, 
                      alpha = 1, #L1 
                      nfolds = 10) #10 fold cross-validation
cat("Cross-validation complete\n")
cat("Optimal lambda:", cv_lasso$lambda.min, "\n") #gives min cross validation error
cat("Lambda 1se:", cv_lasso$lambda.1se, "\n") #largest lambda within 1se 

# 3.Train LASSO model 
model_lasso_1se = glmnet(X_train, y_train, 
                         alpha = 1, 
                         lambda = cv_lasso$lambda.1se)

# 4.Get the coefficients
coef_lasso_1se = coef(model_lasso_1se)
num_nonzero_1se = sum(coef_lasso_1se != 0) - 1

# 5.Generate prediction on train and validation sets
pred_train_lasso_1se = predict(model_lasso_1se, X_train)
pred_val_lasso_1se = predict(model_lasso_1se, X_val)

# 6.Calculate rMSE
rmse_train_lasso_1se = calculate_rMSE(y_train, pred_train_lasso_1se)
rmse_val_lasso_1se = calculate_rMSE(y_val, pred_val_lasso_1se)


cat("\nPerformance Metrics - LASSO (lambda.1se):\n")
cat("Non-zero coefficients:", num_nonzero_1se, "\n")
cat("Training rMSE:", round(rmse_train_lasso_1se, 4), "\n")
cat("Validation rMSE:", round(rmse_val_lasso_1se, 4), "\n")
cat("Difference:", round(rmse_val_lasso_1se - rmse_train_lasso_1se, 4), "\n") #check for overfiting 

# 7.Extract features from LASSO.1se
coefs = coef(model_lasso_1se, s = cv_lasso$lambda.min)
# Convert to a data frame for easier filtering
coef_df = data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)
# 8.Filter for non-zero coefficients (-intercept)
selected_features_df = coef_df %>%
  filter(coefficient != 0, feature != "(Intercept)")
# 9.Extract the list of feature names selected by LASSO
lasso_selected_predictors = selected_features_df$feature
cat("Features selected by LASSO for MLR:", length(lasso_selected_predictors), "predictors\n")

# 10.Combine train and validation sets
full_train = rbind(train_data,validation_data)
cat("Full training data size:", nrow(full_train), "\n") 

# 11.Prepare matrices
X_full = as.matrix(full_train[, -which(names(full_train) == "exam_score")])
y_full = full_train$exam_score

# 12.Model: LASSO (lambda.1se) - retrain
model_lasso_final = glmnet(X_full, y_full, 
                            alpha = 1, 
                            lambda = cv_lasso$lambda.1se)

# Get the coefficients
coef_lasso_final = coef(model_lasso_final)
num_features_lasso = sum(coef_lasso_final != 0) - 1
cat("LASSO features:", num_features_lasso, "\n")
cat("Deviance ratio:", model_lasso_final$dev.ratio, "\n")
cat("Lambda:", model_lasso_final$lambda, "\n")
cat("Non-zero coefficients:", model_lasso_final$df, "\n")

# 13.Generate prediction on test_encoded
# Convert test to matrix for LASSO
X_test = as.matrix(test_encoded)
pred_lasso_test = as.vector(predict(model_lasso_final, X_test))

In [ ]:
# Generate LASSO prediction on full train data 
pred_full_train_lasso = predict(model_lasso_final, 
                                   newx = as.matrix(full_train[, names(full_train) != "exam_score"]), 
                                   s = cv_lasso$lambda.1se)
cat("\nModel trained on full dataset\n")

In [ ]:
# Verification step

cat("=== Data Verification ===\n")
cat("Full training data size:", nrow(full_train), "\n")
cat("Test data size:", nrow(test_encoded), "\n")
cat("\n")

cat("=== Prediction Verification ===\n")
cat("Training predictions length:", length(pred_full_train_lasso), "\n")
cat("Test predictions length:", length(pred_lasso_test), "\n")
cat("\n")

# Check which one matches test data size
if(length(pred_lasso_test) == nrow(test_encoded)) {
    cat("pred_lasso_test matches test data size\n")
    cat("This is the CORRECT prediction for submission\n")
} else {
    warning("Mismatch between pred_lasso_test and test data!")
}

if(length(pred_full_train_lasso) == nrow(full_train)) {
    cat("pred_full_train_lasso matches training data size\n")
    cat("This is for evaluation only, NOT for submission\n")
} else {
    warning("Mismatch between pred_full_train_lasso and training data!")
}

In [ ]:

# Verify we have the correct predictions
cat("=== Generating Submission File ===\n")
cat("Number of test predictions:", length(pred_lasso_test), "\n")
cat("Number of test rows:", nrow(test), "\n")

# Check lengths match
if(length(pred_lasso_test) != nrow(test)) {
    stop("ERROR: Prediction length does not match test data rows!")
}

# Create submission file with original IDs
submission_file1 = data.frame(
    id = test$id,                    # Use original ID from test data
    exam_score = pred_lasso_test     # Your LASSO predictions
)

# Verify submission structure
cat("\n=== Submission File Structure ===\n")
cat("Columns:", paste(names(submission_file1), collapse = ", "), "\n")
cat("Number of rows:", nrow(submission_file1), "\n")
cat("Column types:\n")
print(sapply(submission_file1, class))

# Preview the submission
cat("\n=== Preview First 10 Rows ===\n")
print(head(submission_file1, 10))

cat("\n=== Preview Last 10 Rows ===\n")
print(tail(submission_file1, 10))

# Check for any issues
cat("\n=== Data Quality Checks ===\n")
cat("Any NA in id?", any(is.na(submission_file1$id)), "\n")
cat("Any NA in exam_score?", any(is.na(submission_file1$exam_score)), "\n")
cat("Any duplicate IDs?", any(duplicated(submission_file1$id)), "\n")

# Prediction statistics
cat("\n=== Prediction Statistics ===\n")
cat("Min exam_score:", min(submission_file1$exam_score), "\n")
cat("Max exam_score:", max(submission_file1$exam_score), "\n")
cat("Mean exam_score:", mean(submission_file1$exam_score), "\n")
cat("Median exam_score:", median(submission_file1$exam_score), "\n")

# Write to CSV
write.csv(submission_file1, 
          "RegressionPredictLabel1.csv", 
          row.names = FALSE)

cat("\n✓ Submission file created: RegressionPredictLabel1.csv\n")
cat("✓ File location:", file.path(getwd(), "RegressionPredictLabel1.csv"), "\n")

# Verify the file was written correctly
cat("\n=== Verifying Written File ===\n")
verify_read = read.csv("RegressionPredictLabel1.csv")
cat("File rows:", nrow(verify_read), "\n")
cat("File columns:", paste(names(verify_read), collapse = ", "), "\n")
cat("First few rows from file:\n")
print(head(verify_read, 5))

### XGBoost

In [10]:
install.packages("xgboost",repos='http://cran.us.r-project.org') 
library(xgboost)

Warning message in install.packages("xgboost", repos = "http://cran.us.r-project.org"):
“installation of package ‘xgboost’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done


Attaching package: ‘xgboost’


The following object is masked from ‘package:dplyr’:

    slice




In [12]:
cat("=== Step 1: Prepare Data for XGBoost ===\n")
# train data
X_train = as.matrix(train_data[, -which(names(train_data) == "exam_score")])
y_train = train_data$exam_score

# validation data
X_val = as.matrix(validation_data[, -which(names(validation_data) == "exam_score")])
y_val = validation_data$exam_score

# test data 
X_test = as.matrix(test_encoded)

#full_train = rbind(train_data,validation_data)

# Create DMatrix objects (XGBoost's optimized data structure)
dtrain = xgb.DMatrix(data = X_train, label = y_train)
dval = xgb.DMatrix(data = X_val, label = y_val)
dtest = xgb.DMatrix(data = X_test)

cat("Training matrix:", nrow(X_train), "rows ×", ncol(X_train), "features\n")
cat("Validation matrix:", nrow(X_val), "rows ×", ncol(X_val), "features\n")
cat("Test matrix:", nrow(X_test), "rows ×", ncol(X_test), "features\n")

# Verify no missing values
cat("\nData quality checks:\n")
cat("Missing values in train:", sum(is.na(X_train)), "\n")
cat("Missing values in validation:", sum(is.na(X_val)), "\n")
cat("Missing values in test:", sum(is.na(X_test)), "\n")

cat("\n=== Step 2: Set XGBoost Parameters ===\n")

# Define parameters
params = list(
    objective = "reg:squarederror",  # For regression (minimizes squared error)
    eval_metric = "rmse",            # Evaluation metric
    eta = 0.05,                      # Learning rate (0.01-0.3, smaller = better but slower)
    max_depth = 6,                   # Maximum tree depth (3-10)
    subsample = 0.8,                 # Row sampling (0.5-1.0)
    colsample_bytree = 0.8,          # Column sampling (0.5-1.0)
    min_child_weight = 3,            # Minimum sum of weights in a leaf (1-10)
    gamma = 0 # Minimum loss reduction (0-5, higher = more conservative)
)

cat("Parameters set:\n")
print(params)


cat("\n=== Step 3: Training XGBoost with Validation Monitoring ===\n")

# Create watchlist to monitor both train and validation error
watchlist = list(train = dtrain, validation = dval)

# Train model with early stopping
set.seed(123)
model_xgb = xgb.train(
    params = params,
    data = dtrain,
    nrounds = 1000,                  # Maximum iterations
    watchlist = watchlist,           # Monitor both train and validation
    early_stopping_rounds = 50,      # Stop if validation doesn't improve for 50 rounds
    verbose = 1,                     # Print progress
    print_every_n = 50               # Print every 50 rounds
)

# Get best iteration (where validation error was minimum)
best_iteration = model_xgb$best_iteration
cat("\nTraining complete\n")
cat("Best iteration:", best_iteration, "\n")
cat("Best validation rMSE:", model_xgb$best_score, "\n")

cat("\n=== Step 4: Model Evaluation ===\n")

# Predict on training data
pred_train_xgb = predict(model_xgb, dtrain)
rmse_train_xgb = sqrt(mean((y_train - pred_train_xgb)^2))

# Predict on validation data
pred_val_xgb = predict(model_xgb, dval)
rmse_val_xgb = sqrt(mean((y_val - pred_val_xgb)^2))

# Display results
cat("\nPerformance Metrics - XGBoost:\n")
cat("Training rMSE:", round(rmse_train_xgb, 4), "\n")
cat("Validation rMSE:", round(rmse_val_xgb, 4), "\n")
cat("Difference (Val - Train):", round(rmse_val_xgb - rmse_train_xgb, 4), "\n")

# Check for overfitting
if(rmse_val_xgb - rmse_train_xgb > 0.5) {
    warning("⚠ Possible overfitting detected (large train/val gap)")
} else if(rmse_val_xgb - rmse_train_xgb < 0) {
    cat("✓ No overfitting (validation error ≤ training error)\n")
} else {
    cat("✓ Good generalization (small train/val gap)\n")
}

=== Step 1: Prepare Data for XGBoost ===
Training matrix: 441002 rows × 24 features
Validation matrix: 188998 rows × 24 features
Test matrix: 270000 rows × 24 features

Data quality checks:
Missing values in train: 0 
Missing values in validation: 0 
Missing values in test: 0 

=== Step 2: Set XGBoost Parameters ===
Parameters set:
$objective
[1] "reg:squarederror"

$eval_metric
[1] "rmse"

$eta
[1] 0.05

$max_depth
[1] 6

$subsample
[1] 0.8

$colsample_bytree
[1] 0.8

$min_child_weight
[1] 3

$gamma
[1] 0


=== Step 3: Training XGBoost with Validation Monitoring ===
[1]	train-rmse:61.668192	validation-rmse:61.688927 
Multiple eval metrics are present. Will use validation_rmse for early stopping.
Will train until validation_rmse hasn't improved in 50 rounds.

[51]	train-rmse:10.419315	validation-rmse:10.453226 
[101]	train-rmse:8.851973	validation-rmse:8.899562 
[151]	train-rmse:8.762411	validation-rmse:8.820789 
[201]	train-rmse:8.732859	validation-rmse:8.804840 
[251]	train-rmse:8.71

In [13]:
cat("\n=== Step 5: Retraining on Full Dataset ===\n")

# Combine train and validation for final model
full_train = rbind(train_data, validation_data)
cat("Full training data size:", nrow(full_train), "rows\n")

# Prepare full training matrix
X_full = as.matrix(full_train[, -which(names(full_train) == "exam_score")])
y_full = full_train$exam_score
dtrain_full = xgb.DMatrix(data = X_full, label = y_full)

cat("Retraining with", best_iteration, "rounds (optimal from CV)...\n")

# Train final model using the optimal number of rounds
set.seed(123)
model_xgb_final = xgb.train(
    params = params,
    data = dtrain_full,
    nrounds = best_iteration,        # Use optimal rounds from validation
    verbose = 1
)

cat("✓ Final model training complete\n")

# Evaluate on full training data (just for reference)
pred_full_train = predict(model_xgb_final, dtrain_full)
rmse_full_train = sqrt(mean((y_full - pred_full_train)^2))
cat("Full training rMSE:", round(rmse_full_train, 4), "\n")


=== Step 5: Retraining on Full Dataset ===
Full training data size: 630000 rows
Retraining with 1000 rounds (optimal from CV)...
✓ Final model training complete
Full training rMSE: 8.5317 


In [14]:
cat("\n=== Step 6: Generating Test Predictions ===\n")

# Predict on test data
pred_xgb_test = predict(model_xgb_final, dtest)

# Prediction statistics
cat("\nPrediction Statistics:\n")
cat("Number of predictions:", length(pred_xgb_test), "\n")
cat("Min prediction:", round(min(pred_xgb_test), 2), "\n")
cat("Max prediction:", round(max(pred_xgb_test), 2), "\n")
cat("Mean prediction:", round(mean(pred_xgb_test), 2), "\n")
cat("Median prediction:", round(median(pred_xgb_test), 2), "\n")
cat("Std dev:", round(sd(pred_xgb_test), 2), "\n")

# Check for issues
if(any(is.na(pred_xgb_test))) {
    stop("ERROR: Predictions contain NA values!")
}
if(any(!is.finite(pred_xgb_test))) {
    stop("ERROR: Predictions contain infinite values!")
}

cat("✓ All predictions are valid\n")


=== Step 6: Generating Test Predictions ===

Prediction Statistics:
Number of predictions: 270000 
Min prediction: 7.36 
Max prediction: 106.96 
Mean prediction: 59.78 
Median prediction: 59.89 
Std dev: 17.38 
✓ All predictions are valid


In [15]:
cat("\n=== Step 7: Creating Submission File ===\n")

# Get test IDs from appropriate source
if("id" %in% names(test)) {
    test_ids = test$id
    cat("✓ Using IDs from original test data\n")
} else if("id" %in% names(test_encoded)) {
    test_ids = test_encoded$id
    cat("✓ Using IDs from test_encoded\n")
} else {
    # Create sequential IDs (adjust starting number based on your data)
    test_ids = seq(630000, 630000 + length(pred_xgb_test) - 1)
    cat("⚠ Creating sequential IDs starting from 630000\n")
}

# Verify lengths match
if(length(test_ids) != length(pred_xgb_test)) {
    stop("ERROR: Number of IDs (", length(test_ids), 
         ") doesn't match number of predictions (", length(pred_xgb_test), ")")
}

# Create submission dataframe
submission_xgb = data.frame(
    id = test_ids,
    exam_score = pred_xgb_test
)

# Verify submission structure
cat("\n=== Submission File Verification ===\n")
cat("Columns:", paste(names(submission_xgb), collapse = ", "), "\n")
cat("Number of rows:", nrow(submission_xgb), "\n")
cat("Column types:", paste(sapply(submission_xgb, class), collapse = ", "), "\n")

# Check for duplicates or missing IDs
cat("\nData Quality Checks:\n")
cat("Any duplicate IDs:", any(duplicated(submission_xgb$id)), "\n")
cat("Any NA in id:", any(is.na(submission_xgb$id)), "\n")
cat("Any NA in exam_score:", any(is.na(submission_xgb$exam_score)), "\n")
cat("ID range:", min(submission_xgb$id), "to", max(submission_xgb$id), "\n")
cat("Score range:", round(min(submission_xgb$exam_score), 2), 
    "to", round(max(submission_xgb$exam_score), 2), "\n")

# Preview submission
cat("\n=== Preview First 10 Rows ===\n")
print(head(submission_xgb, 10))

cat("\n=== Preview Last 10 Rows ===\n")
print(tail(submission_xgb, 10))

# Save to CSV
write.csv(submission_xgb, 
          "RegressionPredictLabel_XGBoost.csv", 
          row.names = FALSE)

cat("Submission file created: RegressionPredictLabel_XGBoost.csv\n")
cat("File location:", file.path(getwd(), "RegressionPredictLabel_XGBoost.csv"), "\n")


=== Step 7: Creating Submission File ===
✓ Using IDs from original test data

=== Submission File Verification ===
Columns: id, exam_score 
Number of rows: 270000 
Column types: integer, numeric 

Data Quality Checks:
Any duplicate IDs: FALSE 
Any NA in id: FALSE 
Any NA in exam_score: FALSE 
ID range: 630000 to 899999 
Score range: 7.36 to 106.96 

=== Preview First 10 Rows ===
       id exam_score
1  630000   69.02596
2  630001   67.14574
3  630002   85.63299
4  630003   56.67720
5  630004   44.67625
6  630005   70.12227
7  630006   71.99466
8  630007   59.16291
9  630008   73.19810
10 630009   88.26999

=== Preview Last 10 Rows ===
           id exam_score
269991 899990   59.96880
269992 899991   40.84250
269993 899992   72.78251
269994 899993   85.38830
269995 899994   49.57174
269996 899995   59.99012
269997 899996   36.26125
269998 899997   88.96754
269999 899998   51.36631
270000 899999   65.35068
Submission file created: RegressionPredictLabel_XGBoost.csv
File location: /Users

### XGBoost with Different Parameters

In [16]:
cat("\n=== Step 2: Setting Conservative Parameters ===\n")

params_conservative = list(
    objective = "reg:squarederror",
    eval_metric = "rmse",
    
    # SLOWER LEARNING (key change!)
    eta = 0.03,                      # Reduced from 0.05 → forces simpler model
    
    # SIMPLER TREES (key change!)
    max_depth = 4,                   # Reduced from 6 → prevents complex patterns
    
    # MORE AGGRESSIVE SAMPLING (prevents overfitting)
    subsample = 0.7,                 # Reduced from 0.8 → more randomness
    colsample_bytree = 0.7,          # Reduced from 0.8 → more randomness
    
    # STRONGER REGULARIZATION (key change!)
    min_child_weight = 10,           # Increased from 3 → requires more samples per leaf
    gamma = 1,                       # Increased from 0 → prunes weak splits
    lambda = 5,                      # Increased from 1 → L2 regularization
    alpha = 0.5                      # Added L1 regularization (like LASSO!)
)

cat("Conservative parameters:\n")
print(params_conservative)
cat("\n=== Step 3: Training with Early Stopping ===\n")

watchlist = list(train = dtrain, validation = dval)

set.seed(123)
model_xgb_conservative = xgb.train(
    params = params_conservative,
    data = dtrain,
    nrounds = 2000,                  # Allow more rounds (early stopping will handle it)
    watchlist = watchlist,
    early_stopping_rounds = 30,      # Reduced from 50 → stops sooner
    verbose = 1,
    print_every_n = 50
)

best_iteration = model_xgb_conservative$best_iteration
best_score = model_xgb_conservative$best_score

cat("\n✓ Training complete\n")
cat("Best iteration:", best_iteration, "\n")
cat("Best validation rMSE:", round(best_score, 4), "\n")


=== Step 2: Setting Conservative Parameters ===
Conservative parameters:
$objective
[1] "reg:squarederror"

$eval_metric
[1] "rmse"

$eta
[1] 0.03

$max_depth
[1] 4

$subsample
[1] 0.7

$colsample_bytree
[1] 0.7

$min_child_weight
[1] 10

$gamma
[1] 1

$lambda
[1] 5

$alpha
[1] 0.5


=== Step 3: Training with Early Stopping ===
[1]	train-rmse:63.013791	validation-rmse:63.034537 
Multiple eval metrics are present. Will use validation_rmse for early stopping.
Will train until validation_rmse hasn't improved in 30 rounds.

[51]	train-rmse:17.286196	validation-rmse:17.305830 
[101]	train-rmse:10.077026	validation-rmse:10.101674 
[151]	train-rmse:9.197379	validation-rmse:9.222628 
[201]	train-rmse:8.986639	validation-rmse:9.011890 
[251]	train-rmse:8.891993	validation-rmse:8.916326 
[301]	train-rmse:8.842465	validation-rmse:8.866756 
[351]	train-rmse:8.819772	validation-rmse:8.844394 
[401]	train-rmse:8.808678	validation-rmse:8.834483 
[451]	train-rmse:8.800481	validation-rmse:8.827139 
[5

In [17]:
cat("\n=== Step 4: Evaluation ===\n")

pred_train = predict(model_xgb_conservative, dtrain)
pred_val = predict(model_xgb_conservative, dval)

rmse_train = sqrt(mean((y_train - pred_train)^2))
rmse_val = sqrt(mean((y_val - pred_val)^2))

cat("\nPerformance:\n")
cat("Training rMSE:", round(rmse_train, 4), "\n")
cat("Validation rMSE:", round(rmse_val, 4), "\n")
cat("Difference:", round(rmse_val - rmse_train, 4), "\n")

# Check if improvement over first attempt
if(rmse_val < 8.7321) {
    cat("✓ Better than previous XGBoost (", round(8.7321 - rmse_val, 4), " improvement)\n")
}



=== Step 4: Evaluation ===

Performance:
Training rMSE: 8.6833 
Validation rMSE: 8.7583 
Difference: 0.075 


In [18]:
cat("\n=== Step 5: Retrain on Full Data ===\n")

full_train = rbind(train_data, validation_data)
X_full = as.matrix(full_train[, -which(names(full_train) == "exam_score")])
y_full = full_train$exam_score
dtrain_full = xgb.DMatrix(data = X_full, label = y_full)

cat("Full training size:", nrow(full_train), "rows\n")
cat("Using", best_iteration, "rounds\n")

set.seed(123)
model_xgb_final = xgb.train(
    params = params_conservative,
    data = dtrain_full,
    nrounds = best_iteration,
    verbose = 0
)

cat("✓ Final model trained\n")


=== Step 5: Retrain on Full Data ===
Full training size: 630000 rows
Using 2000 rounds
✓ Final model trained


In [19]:
cat("\n=== Step 6: Generate Test Predictions ===\n")

pred_xgb_test = predict(model_xgb_final, dtest)

# CLIP PREDICTIONS to reasonable range (important!)
# Assuming exam scores are 0-100
pred_xgb_test_clipped = pmax(0, pmin(100, pred_xgb_test))

cat("\nPrediction Statistics (before clipping):\n")
cat("Min:", round(min(pred_xgb_test), 2), "\n")
cat("Max:", round(max(pred_xgb_test), 2), "\n")
cat("Mean:", round(mean(pred_xgb_test), 2), "\n")

cat("\nPrediction Statistics (after clipping):\n")
cat("Min:", round(min(pred_xgb_test_clipped), 2), "\n")
cat("Max:", round(max(pred_xgb_test_clipped), 2), "\n")
cat("Mean:", round(mean(pred_xgb_test_clipped), 2), "\n")

# Count how many were clipped
n_clipped_low = sum(pred_xgb_test < 0)
n_clipped_high = sum(pred_xgb_test > 100)
cat("Predictions clipped:", n_clipped_low + n_clipped_high, 
    "(", n_clipped_low, "below 0,", n_clipped_high, "above 100)\n")


=== Step 6: Generate Test Predictions ===

Prediction Statistics (before clipping):
Min: 11.86 
Max: 107.32 
Mean: 60.63 

Prediction Statistics (after clipping):
Min: 11.86 
Max: 100 
Mean: 60.63 
Predictions clipped: 433 ( 0 below 0, 433 above 100)


In [20]:
cat("\n=== Step 7: Create Submission ===\n")

if("id" %in% names(test)) {
    test_ids = test$id
} else if("id" %in% names(test_encoded)) {
    test_ids = test_encoded$id
} else {
    test_ids = seq(630000, 630000 + length(pred_xgb_test_clipped) - 1)
}

submission_xgb_v2 = data.frame(
    id = test_ids,
    exam_score = pred_xgb_test_clipped
)

cat("Submission rows:", nrow(submission_xgb_v2), "\n")
cat("Preview:\n")
print(head(submission_xgb_v2, 10))

write.csv(submission_xgb_v2, 
          "RegressionPredictLabel_XGBoost_v2.csv", 
          row.names = FALSE)

cat("\n✓ Submission file created: RegressionPredictLabel_XGBoost_v2.csv\n")


=== Step 7: Create Submission ===
Submission rows: 270000 
Preview:
       id exam_score
1  630000   68.81316
2  630001   67.13440
3  630002   86.09581
4  630003   52.89415
5  630004   45.34774
6  630005   70.16889
7  630006   70.80973
8  630007   57.63865
9  630008   75.56014
10 630009   89.76112

✓ Submission file created: RegressionPredictLabel_XGBoost_v2.csv
